In [1]:
import pandas as pd
import numpy as np

In [ ]:
# concat 함수로 병합하기
    # 두 개 이상의 데이터프레임을 하나로 합치는 기능

## 1. column 명이 같은 2개의 데이터프레임 합치기 ###################################
df1 = pd.DataFrame({'key1' : [0,1,2,3,4], 'value1' : ['a', 'b', 'c','d','e']}, index=[0,1,2,3,4])
df2 = pd.DataFrame({'key1' : [3,4,5,6,7], 'value1' : ['c','d','e','f','g']}, index=[3,4,5,6,7])

pd.concat([df1, df2])
pd.concat([df1, df2], axis=1) # 기존 위+아래(행 방향, row 레벨) 합치기에서 왼쪽+오른쪽(열 방향, col 레벨)으로 합치기
pd.concat([df1, df2], ignore_index=True) # 기존 index 무시하고 재배열


## 2. column 명이 다른 2개의 데이터프레임 합치기 ###################################
    # join 방식 사용
    # join 방식은 outer인 경우 합집합, inner인 경우 교집합
df3 = pd.DataFrame({'a':['a0','a1','a2','a3'], 'b':['b0','b1','b2','b3'], 'c':['c0','c1','c2','c3']}, index = [0,1,2,3])
df4 = pd.DataFrame({'a':['a2','a3','a4','a5'], 'c':['c2','c3','c4','c5'], 'd':['d1','d2','d3','d4']}, index = [2,3,4,5])

pd.concat([df3, df4])
pd.concat([df3, df4], axis=1)

,a,b,c,a,c,d
0,a0,b0,c0,NaN,NaN,NaN
1,a1,b1,c1,NaN,NaN,NaN
2,a2,b2,c2,a2,c2,d1
3,a3,b3,c3,a3,c3,d2
4,NaN,NaN,NaN,a4,c4,d3
5,NaN,NaN,NaN,a5,c5,d4


In [ ]:
# merge 함수로 병합하기
    # 특정한 column(key)을 기준으로 병합
    # join 방식 사용
        # how 파라미터를 통해 명시
        # inner: 기본 merge 방법, 일치하는 값이 있는 경우 (Merge할 테이블의 데이터가 모두 있는 경우만 가지고 옴)
        # left: left outer join (왼쪽을 기준으로 오른쪽을 채움, 오른쪽에 데이터가 없으면 NaN)
        # right: right outer join (오른쪽을 기준으로 왼쪽을 채움, 왼쪽에 데이터가 없으면 NaN)
        # outer: full outer join (Left와 Right를 합한 것)

customer = pd.DataFrame({'고객ID' : np.arange(6), 
                    '이름' : ['철수', '영희', '길동', '영수', '수민', '동건'], 
                    '나이' : [40, 20, 21, 30, 31, 18]})

orders = pd.DataFrame({'고객ID' : [1, 1, 2, 2, 2, 3, 3, 1, 4, 9], 
                    '상품명' : ['치약', '칫솔', '이어폰', '헤드셋', '수건', '생수', '수건', '치약', '생수', '케이스'], 
                    '수량' : [1, 2, 1, 1, 3, 2, 2, 3, 2, 1]})
# customer 데이터프레임은 고객ID, 이름, 나이 컬럼으로 구성
# order 데이터프레임은 고객ID, 상품명, 수량 컬럼으로 구성
# 2개의 데이터프레임에 '고객ID' 컬럼이 공통으로 존재하고 있음을 알 수 있음

# customer 데이터프레임의 고객ID는 0,1,2,3,4,5로 구성
# order 데이터프레임의 고객ID는 1,2,3,4,9로 구성
# 2개의 데이터프레임에서 둘 다 존재하는 '고객ID'는 1,2,3,4임을 알 수 있음 (customer는 9가 없고, order는 0,5가 없음)

## 1. merge 함수의 on 옵션
    # join 대상이 되는 column 명시
pd.merge(customer, orders, on='고객ID') # inner 방식(default)으로 합침. 고객ID인 1,2,3,4로 합침. 그래서 0,5,9번은 없음
pd.merge(customer, orders, on='고객ID', how='left') # left 방식으로 합침. 고객ID인 0,1,2,3,4,5로 합침. 그래서 9번은 없음
pd.merge(customer, orders, on='고객ID', how='right') # right 방식으로 합침. 고객ID인 1,2,3,4,9로 합침. 그래서 0,5번은 없음
pd.merge(customer, orders, on='고객ID', how='outer') # outer 방식으로 합침. 고객ID인 0,1,2,3,4,5,9로 합침.

## 문제 1. 가장 많이 팔린 상품명이 무엇인가?
cust_order = pd.merge(customer, orders, on='고객ID', how='right') # right 선택 이유: 팔린 아이템 기준이므로 orders 중심
sum_product = cust_order.groupby('상품명')[['고객ID', '나이', '수량']].sum() # 상품명 기준 합산 (이름은 여러명이 중첩되므로 이쁘게 보이기 위해 제외)
sum_product.sort_values('수량', ascending=False) # 수량 기준 최대값을 확인하기 위해 내림차순 정렬


## 문제 2. 영희가 가장 많이 구매한 상품명은 무엇인가?
cust_order = pd.merge(customer, orders, on='고객ID') # inner 선택 이유: '고객'이 '구매'한 아이템이므로 중첩(inner) 중심
sum_name_product = cust_order.groupby(['이름', '상품명']).sum()
영희_products = sum_name_product.loc['영희'] # 영희가 구매한 상품별 수량
영희_products.sort_values('수량', ascending=False) # 이 중에서 수량 기준 내림차순 정렬



,고객ID,나이,수량
상품명,,,
치약,2,40,4
칫솔,1,20,2
